### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]


class TransferError(Exception):
    pass


class TransferFormatError(TransferError):
    pass


class AccountNotFoundError(TransferError):
    pass


class CurrencyMismatchError(TransferError):
    pass


class InsufficientFundsError(TransferError):
    pass


class TransferAmountError(TransferError):
    pass


def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    # TODO: при ошибке конвертации amount использовать raise ... from ...
    arr = raw.split('|')
    transfer_id, from_user, to_user, amount = arr[0], arr[1], arr[2], arr[3]
    try:
        amount = int(amount)
    except ValueError as e:
        raise TransferAmountError('количество должно быть числом!') from e
    obj = {'transfer_id': transfer_id, 'from_user': from_user, 'to_user': to_user, 'amount': amount}
    return obj


def apply_transfer(transfer, wallets):
    # TODO: проверить существование аккаунтов
    # TODO: запретить перевод самому себе
    # TODO: проверить совпадение валют
    # TODO: проверить баланс отправителя
    # TODO: обновить балансы и вернуть dict результата

    if transfer['from_user'] not in wallets.keys():
        raise AccountNotFoundError(f'Пользователь {transfer['from_user']} не найден!')
    if transfer['to_user'] not in wallets.keys():
        raise AccountNotFoundError(f'Пользователь {transfer['to_user']} не найден!')
    if transfer['from_user'] == transfer['to_user']:
        raise TransferFormatError('нельзя переводить самому себе!')
    if wallets[transfer['from_user']]['currency'] != wallets[transfer['to_user']]['currency']:
        raise CurrencyMismatchError('несовпадение валют!')
    if wallets[transfer['from_user']]['balance'] < transfer['amount']:
        raise InsufficientFundsError('не хватает средств!')
    
    wallets[transfer['from_user']]['balance'] -= transfer['amount']
    wallets[transfer['to_user']]['balance'] += transfer['amount']
    return {'transfer_id': transfer['transfer_id'], "status": 'OK'}

    


def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []
    for s in rows:
        try:
            transfer = parse_transfer(s)
            res = apply_transfer(transfer, wallets)
            successes.append(res)
        except Exception as e:
            errors.append((s, type(e).__name__, e))
    return (successes, errors)



# TODO: вызвать process_batch(rows, wallets)
# TODO: вывести успешные переводы
# TODO: вывести ошибки по типам
# TODO: вывести итоговые балансы
# TODO: найти richest_usd_user

res = process_batch(rows, wallets)
print(res[0])
types_err = {}
for el in res[1]:
    types_err.setdefault(el[1], [])
    types_err[el[1]].append(el)
print(types_err)
m = 0
richest_usd_user = None
for key, val in wallets.items():
    print(f"{key}: {val['balance']}")
    if val['currency'] == 'USD' and val['balance'] > m:
        m = val['balance']
        richest_usd_user = key
print(richest_usd_user)
    



[{'transfer_id': 'TR-100', 'status': 'OK'}, {'transfer_id': 'TR-106', 'status': 'OK'}]
{'InsufficientFundsError': [('TR-101|bob|dave|700', 'InsufficientFundsError', InsufficientFundsError('не хватает средств!'))], 'CurrencyMismatchError': [('TR-102|alice|carol|50', 'CurrencyMismatchError', CurrencyMismatchError('несовпадение валют!'))], 'AccountNotFoundError': [('TR-103|eve|bob|30', 'AccountNotFoundError', AccountNotFoundError('Пользователь eve не найден!'))], 'TransferFormatError': [('TR-104|dave|dave|10', 'TransferFormatError', TransferFormatError('нельзя переводить самому себе!'))], 'TransferAmountError': [('TR-105|bob|alice|abc', 'TransferAmountError', TransferAmountError('количество должно быть числом!'))]}
alice: 1000.0
bob: 550.0
carol: 900.0
dave: 250.0
alice
